<a href="https://colab.research.google.com/github/feronika-lab/Risk-Based-Motor-Insurance-Premium-Pricing-Engine-freMTPL2-with-Tweedie-and-XGBoost/blob/main/02_glm_baseline_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Baseline Model menggunakan GLM.**



In [ ]:
import pandas as pd

try:
    df = pd.read_csv('df_final_clear.csv')
    print("Kolom-kolom dalam DataFrame:")
    print(df.columns.tolist())
except FileNotFoundError:
    print("Error: File 'df_final_clear.csv' tidak ditemukan. Pastikan file sudah diunggah.")
except Exception as e:
    print(f"Terjadi kesalahan: {e}")

Kolom-kolom dalam DataFrame:
['IDpol', 'ClaimNb', 'Exposure', 'VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'VehBrand', 'VehGas', 'Area', 'Density', 'Region', 'ClaimAmount', 'ClaimAmount_Capped', 'LogDensity', 'VehAge_Group', 'Area_Encoded', 'VehBrand_Encoded', 'VehGas_Encoded', 'Region_Encoded', 'VehAge_Group_Encoded']


**1. Mengapa GLM Menjadi "Baseline"?**

Industri asuransi adalah industri yang sangat teregulasi. GLM (Generalized Linear Model) telah menjadi standar selama puluhan tahun karena sifatnya yang transparan. Sebelum mencoba model "canggih" seperti XGBoost, harus punya standar pembanding untuk mengetahui: "Apakah model kompleks saya benar-benar lebih baik daripada rumus standar yang sudah ada?"

**2. Tweedie: Solusi **

Seperti yang kita bahas sebelumnya, data asuransi memiliki banyak nilai nol (tidak klaim) dan nilai besar (klaim).

    Daripada membuat dua model terpisah (satu untuk frekuensi dan satu untuk keparahan), Tweedie Regression memungkinkan Anda memodelkan Pure Premium secara langsung dalam satu rumus.

    Secara matematis, ini bekerja karena Tweedie memiliki fungsi varians:
    Var(Y)=ϕμp


    Di mana 1<p<2 menggabungkan karakteristik distribusi Poisson dan Gamma sekaligus.

**3. Significance Testing (P-Value & Koefisien)**

Dalam GLM, kita tidak hanya mendapatkan prediksi harga, tapi juga bukti statistik:

    Koefisien: Menunjukkan seberapa besar pengaruh sebuah fitur. Misal, jika koefisien DrivAge negatif, berarti semakin tua pengemudi, premi cenderung turun.

    P-Value: Menunjukkan apakah fitur tersebut benar-benar berpengaruh atau hanya "kebetulan" saja. Jika P-Value <0.05, maka fitur tersebut dianggap signifikan secara statistik untuk menentukan harga.

Apa yang Akan didapatkan di Tahap Ini?

Di akhir tahap ini, akan memiliki sebuah "Tabel Tarif" sederhana yang bisa dijelaskan kepada siapa pun. "Menurut model standar ini, setiap kenaikan 1 poin BonusMalus akan menaikkan harga premi sebesar X%."

`02_glm_baseline.ipynb` adalah melakukan Significance Testing (melihat p-value dan koefisien), kita akan menggunakan library statsmodels. Library ini jauh lebih lengkap daripada scikit-learn untuk urusan laporan statistik formal.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ==========================================
# 1. DATA LOADING
# ==========================================
print("Memuat dataset df_final_clear.csv...")
df = pd.read_csv('df_final_clear.csv')

# Membuat Target Variable: Pure Premium
# Karena model memprediksi biaya per satuan exposure
df['PurePremium_Capped'] = df['ClaimAmount_Capped'] / df['Exposure']

# ==========================================
# 2. SETUP FORMULA GLM
# ==========================================
# Kita gunakan C(nama_kolom) untuk memberi tahu model bahwa itu adalah variabel Kategorikal (One-Hot Encoding)
# Catatan: VehBrand dan Region tidak kita masukkan ke Baseline karena
# kategorinya terlalu banyak dan bisa membuat GLM standar mengalami overfit/error memori.

formula = """PurePremium_Capped ~ BonusMalus +
                                  DrivAge +
                                  VehPower +
                                  LogDensity +
                                  C(Area) +
                                  C(VehGas) +
                                  C(VehAge_Group)"""

print("\nFormula GLM yang digunakan:")
print(formula)

# ==========================================
# 3. TRAINING MODEL GLM TWEEDIE
# ==========================================
print("\n⏳ Sedang melatih model GLM Tweedie (p=1.5)... Ini mungkin memakan waktu beberapa menit.")

# var_power=1.5 adalah standar industri untuk Premium Pricing (Compound Poisson-Gamma)
# var_weights=df['Exposure'] memastikan bahwa polis yang aktif 1 tahun punya bobot lebih kuat dari polis 1 bulan
tweedie_family = sm.families.Tweedie(link=sm.families.links.Log(), var_power=1.5)

glm_model = smf.glm(formula=formula,
                    data=df,
                    family=tweedie_family,
                    var_weights=df['Exposure'])

glm_result = glm_model.fit()

# ==========================================
# 4. SIGNIFICANCE TESTING & EVALUATION
# ==========================================
print("\n" + "="*70)
print("📊 GLM TWEEDIE MODEL SUMMARY (ACTUARIAL BASELINE)")
print("="*70)
print(glm_result.summary())

# Menghitung prediksi di dataset
df['GLM_Prediction'] = glm_result.predict(df)
print("\n✅ Training Selesai. Prediksi GLM telah ditambahkan ke kolom 'GLM_Prediction'.")
df[['IDpol', 'Exposure', 'PurePremium_Capped', 'GLM_Prediction']].head()

Memuat dataset df_final_clear.csv...

Formula GLM yang digunakan:
PurePremium_Capped ~ BonusMalus + 
                                  DrivAge + 
                                  VehPower + 
                                  LogDensity + 
                                  C(Area) + 
                                  C(VehGas) + 
                                  C(VehAge_Group)

⏳ Sedang melatih model GLM Tweedie (p=1.5)... Ini mungkin memakan waktu beberapa menit.

📊 GLM TWEEDIE MODEL SUMMARY (ACTUARIAL BASELINE)
                 Generalized Linear Model Regression Results                  
Dep. Variable:     PurePremium_Capped   No. Observations:                10934
Model:                            GLM   Df Residuals:                    10920
Model Family:                 Tweedie   Df Model:                           13
Link Function:                    Log   Scale:                          274.85
Method:                          IRLS   Log-Likelihood:                -4539.1
Date:

,IDpol,Exposure,PurePremium_Capped,GLM_Prediction
0,1.0,0.10,0.0,48.703628
1,3.0,0.77,0.0,48.703628
2,5.0,0.75,0.0,50.721589
3,10.0,0.09,0.0,55.177011
4,11.0,0.84,0.0,55.177011


Berikut adalah penjelasan fungsi dan logika bisnis dari setiap blok kode:

**Meyiapkan Target**
    
    df['PurePremium_Capped'] = df['ClaimAmount_Capped'] / df['Exposure']

Baris kedua adalah inti dari Premium Pricing. Kita tidak memprediksi ClaimAmount mentah, melainkan Pure Premium (Premi Murni). Mengapa dibagi Exposure? Ini untuk "menyetahunkan" risiko.

  Contoh: Jika Nasabah A kecelakaan habis 500 Euro tapi dia baru berasuransi 0.5 tahun (6 bulan), maka risiko tahunannya (Pure Premium) sebenarnya adalah 500/0.5=1000 Euro/tahun. Model harus belajar dari nilai 1000 ini agar adil.

**SETUP FORMULA GLM (Menerjemahkan Logika ke Mesin)**

    formula = """PurePremium_Capped ~ BonusMalus + DrivAge + VehPower + LogDensity + C(Area) + C(VehGas) + C(VehAge_Group)"""


Fungsi: Mendefinisikan persamaan matematika untuk GLM. Simbol ~ berarti "diprediksi oleh".

Mengapa pakai C(...)? Ini adalah cara statsmodels melakukan One-Hot Encoding secara otomatis di latar belakang. Saat membangun fondasi Pricing Engine, model GLM yang bersifat linear tidak boleh disuapi data Label Encoding (0, 1, 2, 3...) untuk kategori seperti Area atau VehGas, karena model akan salah mengira bahwa "Area C" nilainya tiga kali lipat "Area A". Dengan C(), model memperlakukan mereka sebagai kategori yang setara.

Mengapa VehBrand dan Region tidak dimasukkan? Karena kategorinya terlalu banyak (kardinalitas tinggi). Memasukkannya ke GLM standar akan membuat tabel prediksi terlalu lebar dan rentan overfitting. Fitur kompleks ini biasanya kita simpan untuk "dilahap" oleh algoritma Gradient Boosting atau Explainable Boosting Machine (EBM) di tahap selanjutnya.

**TRAINING MODEL GLM TWEEDIE (Proses Belajar)**

    tweedie_family = sm.families.Tweedie(link=sm.families.links.Log(), var_power=1.5)
    glm_model = smf.glm(formula=formula, data=df, family=tweedie_family, var_weights=df['Exposure'])
    glm_result = glm_model.fit()


Fungsi: Melatih model menggunakan metode regresi yang didesain khusus untuk data asuransi.

var_power=1.5: Ini adalah pengaturan spesifik Tweedie yang menggabungkan probabilitas "seberapa sering klaim" (Poisson) dan "seberapa besar klaimnya" (Gamma). Parameter 1.5 adalah standar emas di industri aktuaria untuk memodelkan premi murni.

link=Log(): Memastikan hasil tebakan premi selalu positif. Tidak peduli seberapa aman pengemudinya, perusahaan asuransi tidak mungkin memberikan harga premi minus (memberi uang ke nasabah).

var_weights=df['Exposure']: Ini memberi tahu model: "lebih menekankan (gives more weight to) pada data nasabah yang sudah berasuransi 1 tahun penuh (bobot 1.0) lebih daripada nasabah yang baru berasuransi 1 minggu (bobot 0.02)."

**SIGNIFICANCE TESTING & EVALUATION (Rapor Ujian Model)**

    print(glm_result.summary())
    df['GLM_Prediction'] = glm_result.predict(df)

Fungsi: Mencetak laporan statistik formal dan menyimpan hasil tebakan model ke dalam kolom baru GLM_Prediction.

  Mengapa summary() Output ini menjadi bagian kunci dalam proses review, validasi model, dan dokumentasi—baik untuk kebutuhan regulator maupun review internal oleh aktuaris senior. Ringkasan ini menunjukkan dua hal krusial:

  P-Value (P>|z|): Uji signifikansi. Jika nilai P-Value sebuah variabel (misalnya LogDensity) kurang dari 0.05, kita bisa dengan yakin berkata, "Kepadatan wilayah benar-benar mempengaruhi risiko kecelakaan, bukan sekadar kebetulan statistik."

  Koefisien (coef): Besaran dampak. Jika BonusMalus punya koefisien positif, artinya makin tinggi malus, makin mahal harganya.

Hasil dari tahap ini memberikan Anda sebuah Baseline—garis dasar akurasi yang transparan dan bisa dipertanggungjawabkan.

In [ ]:
# Menyimpan dataset beserta hasil prediksi GLM ke file baru
# index=False agar tidak ada kolom nomor baris tambahan yang masuk
df.to_csv('df_scored_glm.csv', index=False)
print("Data beserta prediksi GLM berhasil disimpan ke 'df_scored_glm.csv'")

Data beserta prediksi GLM berhasil disimpan ke 'df_scored_glm.csv'


Catatan Tambahan untuk README GitHub

  "Dalam membangun Baseline Model, kami menggunakan GLM dengan distribusi Tweedie (p=1.5) dan Log-link function. Model ini dilatih dengan Exposure sebagai bobot (weights) untuk menyesuaikan durasi risiko. Fitur kategorikal dengan kardinalitas tinggi seperti Region dan VehBrand sengaja dieksklusi dari baseline model untuk menjaga parsimony (kesederhanaan) model linear, dan nantinya akan dieksploitasi lebih jauh menggunakan model Machine Learning berbasis pohon (XGBoost/EBM)."